# Hands-On 4: SVM and neural-network boundaries

Record a prediction before running each experiment.

In [ ]:
from pathlib import Path
import os
import sys
try:
    import mlcourse.setup
except ModuleNotFoundError:
    bases = [Path(os.environ.get("MLCOURSE_ROOT", Path.cwd())), Path.cwd(), Path("/content/pp-machine-learning")]
    for base in bases:
        for candidate in (base.resolve(), *base.resolve().parents):
            if (candidate / "src/mlcourse/setup.py").is_file():
                sys.path.insert(0, str(candidate / "src"))
                break
        else:
            continue
        break
    else:
        raise RuntimeError("Course files not found. Open the extracted course repository or set MLCOURSE_ROOT to its location.") from None

In [ ]:
from mlcourse.setup import setup_notebook
REPO_ROOT = setup_notebook()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from mlcourse.labs import load_course_data

from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from mlcourse.labs import compare_boundaries, split_classification
from mlcourse.widgets import interactive_svm, interactive_mlp

## 1. Inspect the feature space

Load `ds1`, `ds2` and `ds3`. Use `ds2` for the subsequent experiments.

**Prediction:** Which geometries appear difficult for a single straight boundary?

*Your response.*

In [ ]:
datasets = {name: load_course_data(name) for name in ['ds1', 'ds2', 'ds3']}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), layout='constrained')
for ax, (name, frame) in zip(axes, datasets.items()):
    for label, marker in zip(sorted(frame.Class.unique()), ['o', '^', 's']):
        part = frame[frame.Class == label]
        ax.scatter(part.X1, part.X2, label=str(label), marker=marker, s=22)
    ax.set(title=name, xlabel='X1', ylabel='X2')
    ax.legend(title='Class')
frame = datasets['ds2']
X_train, X_test, y_train, y_test = split_classification(frame[['X1', 'X2']], frame.Class)
plt.show()

**Observation:** Locate overlapping or curved class regions.

*Your response.*

**Explanation:** How can feature-space geometry favour different inductive biases?

*Your response.*

## 2. Compare three kernels

Fit linear, polynomial and RBF SVMs on the same split.

**Prediction:** Which kernels permit curved boundaries?

*Your response.*

In [ ]:
display(compare_boundaries({kernel: make_pipeline(StandardScaler(), SVC(kernel=kernel, C=1, gamma=1, degree=3))
                           for kernel in ['linear', 'poly', 'rbf']}, X_train, X_test, y_train, y_test))

**Observation:** Compare boundary shapes and test accuracy.

*Your response.*

**Explanation:** How does the kernel change the available boundary family?

*Your response.*

## 3. Manipulate the SVM

Change the kernel and its active controls. Compare small and large `C`; for RBF, change `gamma`.

**Prediction:** How might a larger RBF `gamma` change local sensitivity?

*Your response.*

In [ ]:
svm_lab = interactive_svm(X_train, X_test, y_train, y_test)
display(svm_lab.widget)

**Observation:** Record one broad boundary and one more local boundary.

*Your response.*

**Explanation:** Distinguish the effects of `C`, `gamma` and polynomial `degree`.

*Your response.*

## 4. Fit a small neural network

Fit one hidden layer with five units and `activation="tanh"`.

**Prediction:** Can this network form a curved boundary?

*Your response.*

In [ ]:
display(compare_boundaries({'MLP, five units': make_pipeline(StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(5,), activation='tanh', alpha=.01, solver='lbfgs',
                  max_iter=1500, tol=1e-3, random_state=0))}, X_train, X_test, y_train, y_test))

**Observation:** Compare its geometry with the kernel examples.

*Your response.*

**Explanation:** Why does a nonlinear hidden layer change the boundary family?

*Your response.*

## 5. Change capacity and initialisation

Vary hidden-layer width, count and activation. Then keep those fixed and change `random_state`.

**Prediction:** Should repeated initialisations produce exactly the same boundary?

*Your response.*

In [ ]:
mlp_lab = interactive_mlp(X_train, X_test, y_train, y_test)
display(mlp_lab.widget)

**Observation:** Record one capacity change and one seed change.

*Your response.*

**Explanation:** Separate representational capacity, regularisation and optimisation variability.

*Your response.*